# NB03 — Linear Regression from Scratch (Pure Python)

> **Implement every formula by hand. No sklearn, no numpy linalg.**

Build the entire OLS pipeline step by step.


In [ ]:
# Pure Python — no imports needed for the core class
class OLSRegression:
    """Simple linear regression built from scratch."""

    def __init__(self):
        self.slope     = None
        self.intercept = None
        self._fitted   = False

    # ── fitting ──────────────────────────────────────────────────────────────

    def fit(self, X, y):
        n = len(X)
        assert n >= 2, "Need at least 2 data points"
        assert len(y) == n, "X and y must have same length"

        x_mean = sum(X) / n
        y_mean = sum(y) / n

        # β₁ = Σ(xᵢ−x̄)(yᵢ−ȳ) / Σ(xᵢ−x̄)²
        numerator   = sum((X[i] - x_mean) * (y[i] - y_mean) for i in range(n))
        denominator = sum((X[i] - x_mean) ** 2 for i in range(n))

        if denominator == 0:
            raise ValueError("All X values are identical — cannot fit a line.")

        self.slope     = numerator / denominator
        self.intercept = y_mean - self.slope * x_mean
        self._fitted   = True
        return self

    # ── prediction ───────────────────────────────────────────────────────────

    def predict(self, X):
        assert self._fitted, "Call fit() first"
        return [self.intercept + self.slope * x for x in X]

    # ── metrics ──────────────────────────────────────────────────────────────

    def residuals(self, X, y):
        y_pred = self.predict(X)
        return [y[i] - y_pred[i] for i in range(len(y))]

    def ssr(self, X, y):
        res = self.residuals(X, y)
        return sum(r ** 2 for r in res)

    def mse(self, X, y):
        return self.ssr(X, y) / len(y)

    def rmse(self, X, y):
        return self.mse(X, y) ** 0.5

    def r_squared(self, X, y):
        n = len(y)
        y_mean  = sum(y) / n
        ss_tot  = sum((yi - y_mean) ** 2 for yi in y)
        ss_res  = self.ssr(X, y)
        if ss_tot == 0:
            return float('nan')
        return 1 - ss_res / ss_tot

    # ── standard errors & inference ──────────────────────────────────────────

    def standard_errors(self, X, y):
        """SE of intercept and slope."""
        n       = len(y)
        res     = self.residuals(X, y)
        sigma2  = sum(r**2 for r in res) / (n - 2)   # unbiased: divide by n-2
        x_mean  = sum(X) / n
        sxx     = sum((xi - x_mean) ** 2 for xi in X)
        se_b1   = (sigma2 / sxx) ** 0.5
        se_b0   = (sigma2 * (1/n + x_mean**2 / sxx)) ** 0.5
        return se_b0, se_b1

    def t_statistics(self, X, y):
        se_b0, se_b1 = self.standard_errors(X, y)
        t_b0 = self.intercept / se_b0
        t_b1 = self.slope     / se_b1
        return t_b0, t_b1

    def summary(self, X, y):
        se_b0, se_b1 = self.standard_errors(X, y)
        t_b0, t_b1   = self.t_statistics(X, y)
        r2            = self.r_squared(X, y)
        print(f"{'='*55}")
        print(f"  OLS Regression Results")
        print(f"{'='*55}")
        print(f"  {'Coef':>10}  {'SE':>10}  {'t':>10}")
        print(f"  {'-'*40}")
        print(f"  intercept  {self.intercept:>10.4f}  {se_b0:>10.4f}  {t_b0:>10.4f}")
        print(f"  slope      {self.slope:>10.4f}  {se_b1:>10.4f}  {t_b1:>10.4f}")
        print(f"{'='*55}")
        print(f"  R²  = {r2:.6f}")
        print(f"  MSE = {self.mse(X,y):.6f}")
        print(f"  RMSE= {self.rmse(X,y):.6f}")
        print(f"  SSR = {self.ssr(X,y):.6f}")
        print(f"  n   = {len(y)}")


# ── test it ──────────────────────────────────────────────────────────────────
X = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
y = [40, 45, 50, 55, 65, 70, 75, 85, 90, 95]

model = OLSRegression()
model.fit(X, y)
model.summary(X, y)


In [ ]:
# Cross-check against sklearn
from sklearn.linear_model import LinearRegression as SklearnLR
import numpy as np

X_arr = np.array(X).reshape(-1, 1)
y_arr = np.array(y)

sk = SklearnLR().fit(X_arr, y_arr)
print(f"sklearn  slope={sk.coef_[0]:.6f}  intercept={sk.intercept_:.6f}")
print(f"scratch  slope={model.slope:.6f}  intercept={model.intercept:.6f}")
print(f"Match: slope={abs(sk.coef_[0]-model.slope)<1e-8}, intercept={abs(sk.intercept_-model.intercept)<1e-8}")


In [ ]:
# Manual step-by-step table — shows every arithmetic step
import numpy as np
import pandas as pd

X = np.array([1.,2.,3.,4.,5.])
y = np.array([2.,4.,5.,4.,5.])
xbar, ybar = X.mean(), y.mean()

rows = []
for xi, yi in zip(X, y):
    rows.append({
        'x': xi, 'y': yi,
        'x - x̄': xi - xbar,
        'y - ȳ': yi - ybar,
        '(x-x̄)(y-ȳ)': (xi-xbar)*(yi-ybar),
        '(x-x̄)²': (xi-xbar)**2,
    })

df = pd.DataFrame(rows)
sums = df.sum(numeric_only=True)

print(df.to_string(index=False))
print("\nSums:")
print(f"  Σ(x-x̄)(y-ȳ) = {sums['(x-x̄)(y-ȳ)']:.2f}")
print(f"  Σ(x-x̄)²     = {sums['(x-x̄)²']:.2f}")
b1 = sums['(x-x̄)(y-ȳ)'] / sums['(x-x̄)²']
b0 = ybar - b1 * xbar
print(f"\nβ₁ = {sums['(x-x̄)(y-ȳ)']:.2f} / {sums['(x-x̄)²']:.2f} = {b1:.4f}")
print(f"β₀ = {ybar:.2f} - {b1:.4f}×{xbar:.2f} = {b0:.4f}")


## Key takeaways

- The entire algorithm is ~20 lines of pure Python
- Standard errors need `n-2` in the denominator (2 parameters estimated)
- `t = coef / SE` — measures how many standard errors away from zero
- Cross-check always: scratch vs sklearn results must match to 8 decimal places

**Next:** NB04 — R², residuals, and sum-of-squares decomposition explained visually.
